# Attentionメカニズム - 対話的学習ノートブック

このノートブックでは、Attentionメカニズムについて段階的に学習します。

## 目次
1. **セクション1**: ライブラリインポートとセットアップ
2. **セクション2**: 基本概念の理解
3. **セクション3**: Scaled Dot-Product Attention の実装
4. **セクション4**: Multi-Head Attention の実装
5. **セクション5**: Attention の可視化
6. **セクション6**: Transformer ブロック
7. **セクション7**: 実践的な例

---

**所要時間**: 約 2-3 時間

**推奨環境**: Python 3.8+, PyTorch 1.9+, NumPy 1.19+

## セクション1: ライブラリインポートとセットアップ

まず必要なライブラリをインポートし、動作確認を行います。

In [1]:
# 基本的なライブラリ
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語フォント設定
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import MultiheadAttention as TorchMHA

# ユーティリティ
import math
from typing import Tuple, Optional

# 表示設定
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# PyTorchの警告を最小化
import warnings
warnings.filterwarnings('ignore')

print("✅ ライブラリインポート完了")
print(f"PyTorch バージョン: {torch.__version__}")
print(f"NumPy バージョン: {np.__version__}")

ImportError: /home/abemc/.pyenv/versions/3.10.13/lib/python3.10/site-packages/torch/lib/libtorch_cuda.so: undefined symbol: ncclCommResume

In [ ]:
# GPU/CPU 設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

# 乱数シード設定（再現性のため）
torch.manual_seed(42)
np.random.seed(42)

print("✅ セットアップ完了")

## セクション2: 基本概念の理解

### Attention の基本公式

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**各要素の説明:**
- **Q (Query)**: クエリ行列 - 現在のトークンが「何に注目するか」を表現
- **K (Key)**: キー行列 - 他のトークンの「特徴」を表現
- **V (Value)**: バリュー行列 - 他のトークンの「情報」を表現
- **$d_k$**: キーの次元 - スケーリング係数
- **softmax**: 確率分布への変換

In [ ]:
# 簡単な例で理解してみる

# テキスト例
sentence = "I love cats"
tokens = sentence.split()
seq_len = len(tokens)
d_model = 4  # 埋め込み次元（簡単のため小さく）

print(f"文章: {sentence}")
print(f"トークン数: {seq_len}")
print(f"埋め込み次元: {d_model}")
print()

# ダミー埋め込み（実際には学習済みモデルから得る）
embeddings = torch.randn(seq_len, d_model)

print("各トークンの埋め込みベクトル:")
for i, token in enumerate(tokens):
    print(f"{token:6s}: {embeddings[i].tolist()}")

In [ ]:
# Self-Attention の計算手順を詳しく見る
Q = embeddings.clone()  # Query
K = embeddings.clone()  # Key
V = embeddings.clone()  # Value

print("Step 1: スコア行列の計算 (QK^T)")
scores = torch.matmul(Q, K.T)
print(f"形状: {scores.shape}")
print("スコア行列:")
print(scores)
print()

# スコア行列の解釈: 各行 i は、トークン i がどのトークンにどれだけ注目するかを表す
print("解釈: 各行は各トークンの注目パターンを表します")
for i, token in enumerate(tokens):
    scores_i = scores[i].tolist()
    print(f"{token:6s} の注目スコア: {[f'{s:.2f}' for s in scores_i]}")

In [ ]:
# Step 2: スケーリング
d_k = Q.shape[-1]
print(f"Step 2: スケーリング (√d_k = √{d_k} = {math.sqrt(d_k):.2f})")
scaled_scores = scores / math.sqrt(d_k)
print("スケーリング後のスコア:")
print(scaled_scores)
print()

# なぜスケーリング?
print("スケーリングの理由:")
print(f"  - 勾配爆発を防ぐ")
print(f"  - softmax の入力を安定させる")
print(f"  - 学習効率を向上させる")

In [ ]:
# Step 3: softmax で確率分布に変換
print("Step 3: softmax を適用")
attention_weights = F.softmax(scaled_scores, dim=-1)
print("Attention ウェイト (確率分布):")
print(attention_weights)
print()

# 各行の合計が1であることを確認
print("各行の合計 (確認: すべて1になるはず):")
print(attention_weights.sum(dim=-1))
print()

# 解釈
print("解釈: 各トークンの注目分布 (パーセンテージ)")
for i, token in enumerate(tokens):
    weights = (attention_weights[i] * 100).tolist()
    print(f"{token:6s}: {tokens[0]} {weights[0]:5.1f}% | {tokens[1]} {weights[1]:5.1f}% | {tokens[2]} {weights[2]:5.1f}%")

In [ ]:
# Step 4: Value との乗算
print("Step 4: Value との乗算で最終出力を計算")
output = torch.matmul(attention_weights, V)
print(f"出力形状: {output.shape}")
print("出力ベクトル:")
for i, token in enumerate(tokens):
    print(f"{token:6s}: {output[i].tolist()}")
print()
print("✅ Self-Attention の計算完了")

## セクション3: Scaled Dot-Product Attention の実装

In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    mask: Optional[torch.Tensor] = None
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Scaled Dot-Product Attention を計算します。
    
    Args:
        Q: Query 行列 (batch_size, seq_len, d_k)
        K: Key 行列 (batch_size, seq_len, d_k)
        V: Value 行列 (batch_size, seq_len, d_v)
        mask: 注目マスク (オプション)
    
    Returns:
        output: Attention 出力 (batch_size, seq_len, d_v)
        attention_weights: Attention ウェイト (batch_size, seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # スコア計算
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # マスク適用（オプション）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # Value との乗算
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

print("✅ scaled_dot_product_attention 関数を定義")

In [ ]:
# 実装のテスト
batch_size = 2
seq_len = 4
d_k = 8

Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Query 形状: {Q.shape}")
print(f"Key 形状: {K.shape}")
print(f"Value 形状: {V.shape}")
print()
print(f"出力形状: {output.shape}")
print(f"Attention ウェイト形状: {weights.shape}")
print()
print("最初のバッチの Attention ウェイト:")
print(weights[0])

## セクション4: Multi-Head Attention の実装

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention の実装"""
    
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model は num_heads で割り切れる必要があります"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 線形変換層
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]
        
        # 線形変換
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)
        
        # マルチヘッドに分割
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Attention 計算
        output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # ヘッドを統合
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, -1, self.d_model)
        
        # 最終線形変換
        output = self.W_o(output)
        
        return output, attention_weights

print("✅ MultiHeadAttention クラスを定義")

In [ ]:
# Multi-Head Attention のテスト
batch_size = 2
seq_len = 5
d_model = 32
num_heads = 4

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# テスト入力
x = torch.randn(batch_size, seq_len, d_model)

# 順伝播
output, attention_weights = mha(x, x, x)

print(f"入力形状: {x.shape}")
print(f"出力形状: {output.shape}")
print(f"Attention ウェイト形状: {attention_weights.shape}")
print()
print(f"各ヘッドの次元: d_k = {d_model} / {num_heads} = {d_model // num_heads}")
print()
print("✅ Multi-Head Attention は正常に動作")

## セクション5: Attention の可視化

In [ ]:
def visualize_attention_weights(weights: torch.Tensor, tokens: list, title: str = "Attention Weights"):
    """
    Attention ウェイトをヒートマップで可視化
    
    Args:
        weights: Attention ウェイト (seq_len, seq_len) または (batch, seq_len, seq_len)
        tokens: トークンリスト
        title: グラフのタイトル
    """
    # バッチ次元を除去
    if weights.dim() == 3:
        weights = weights[0]  # 最初のサンプルのみ
    
    # NumPy に変換
    weights_np = weights.detach().numpy()
    
    # プロット
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(weights_np, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=tokens, yticklabels=tokens,
                cbar_kws={"label": "Attention Weight"}, ax=ax)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel("Key (注目対象)", fontsize=12)
    ax.set_ylabel("Query (注目元)", fontsize=12)
    
    plt.tight_layout()
    plt.show()

print("✅ 可視化関数を定義")

In [ ]:
# 簡単な例で可視化
sentence = "I love cats"
tokens = sentence.split()

# Attention 計算
embeddings = torch.randn(len(tokens), 8)
output, weights = scaled_dot_product_attention(
    embeddings.unsqueeze(0),
    embeddings.unsqueeze(0),
    embeddings.unsqueeze(0)
)

# 可視化
visualize_attention_weights(weights, tokens, title="Self-Attention Weights: 'I love cats'")

## セクション6: Transformer ブロック

In [ ]:
class TransformerBlock(nn.Module):
    """Transformer ブロック (Attention + FFN)"""
    
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        
        # Multi-Head Attention
        self.attention = MultiHeadAttention(d_model, num_heads)
        
        # Feed-Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        
        # Layer Normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # 残差接続 + Layer Norm
        attn_output, _ = self.attention(x, x, x, mask)
        x = x + self.dropout(attn_output)
        x = self.norm1(x)
        
        # FFN + 残差接続 + Layer Norm
        ffn_output = self.ffn(x)
        x = x + self.dropout(ffn_output)
        x = self.norm2(x)
        
        return x

print("✅ TransformerBlock クラスを定義")

In [ ]:
# Transformer ブロックのテスト
batch_size = 2
seq_len = 5
d_model = 32
num_heads = 4
d_ff = 128

transformer_block = TransformerBlock(d_model, num_heads, d_ff)

# テスト入力
x = torch.randn(batch_size, seq_len, d_model)

# 順伝播
output = transformer_block(x)

print(f"入力形状: {x.shape}")
print(f"出力形状: {output.shape}")
print()
print("Transformer ブロックのコンポーネント:")
print(f"  - Multi-Head Attention: {num_heads} ヘッド")
print(f"  - Feed-Forward: {d_model} -> {d_ff} -> {d_model}")
print(f"  - Layer Normalization: 2層")
print(f"  - 残差接続: 2箇所")
print()
print("✅ Transformer ブロックは正常に動作")

## セクション7: 実践的な例

簡単なシーケンス分類タスク（テキスト分類）を実装します。

In [ ]:
class SimpleTransformer(nn.Module):
    """シンプルな Transformer モデル"""
    
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, num_layers: int, num_classes: int):
        super().__init__()
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional Encoding（簡略版）
        self.pos_encoding = nn.Parameter(torch.randn(1, 100, d_model))  # Max seq_len = 100
        
        # Transformer ブロック
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_model * 4)
            for _ in range(num_layers)
        ])
        
        # 分類層
        self.fc = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        # Embedding
        x = self.embedding(x)
        
        # Positional Encoding を加算
        x = x + self.pos_encoding[:, :x.shape[1], :]
        
        # Transformer ブロックを通す
        for block in self.transformer_blocks:
            x = block(x)
        
        # 平均プール
        x = x.mean(dim=1)
        
        # 分類
        x = self.fc(x)
        
        return x

print("✅ SimpleTransformer クラスを定義")

In [ ]:
# モデルの初期化と試行
vocab_size = 1000
d_model = 64
num_heads = 4
num_layers = 2
num_classes = 2  # バイナリ分類

model = SimpleTransformer(vocab_size, d_model, num_heads, num_layers, num_classes)

# ダミー入力（単語 ID のシーケンス）
batch_size = 4
seq_len = 10
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

# 順伝播
logits = model(input_ids)

print(f"入力形状 (単語 ID): {input_ids.shape}")
print(f"出力形状 (ロジット): {logits.shape}")
print()
print("モデルアーキテクチャ:")
print(f"  - 語彙サイズ: {vocab_size}")
print(f"  - 埋め込み次元: {d_model}")
print(f"  - ヘッド数: {num_heads}")
print(f"  - Transformer ブロック層数: {num_layers}")
print(f"  - 分類クラス数: {num_classes}")
print()
print("✅ モデルは正常に動作")

In [ ]:
# パラメータ数の計算
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"総パラメータ数: {total_params:,}")
print(f"学習可能なパラメータ数: {trainable_params:,}")

# 各層のパラメータ数
print("\n各層のパラメータ数:")
for name, module in model.named_modules():
    if isinstance(module, (nn.Linear, nn.Embedding, nn.LayerNorm)):
        params = sum(p.numel() for p in module.parameters())
        if params > 0:
            print(f"  {name:30s}: {params:8,}")

## セクション8: 学習チュートリアル

実際にモデルを学習させてみましょう。

In [ ]:
# 学習ループの例
def train_step(model, batch_input_ids, batch_labels, optimizer, criterion):
    """1ステップの学習"""
    optimizer.zero_grad()
    
    # 順伝播
    logits = model(batch_input_ids)
    loss = criterion(logits, batch_labels)
    
    # 逆伝播
    loss.backward()
    optimizer.step()
    
    return loss.item()

# モデルの初期化
model = SimpleTransformer(vocab_size=1000, d_model=64, num_heads=4, num_layers=2, num_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# ダミーデータでの学習
print("学習を開始します...\n")
losses = []

for epoch in range(5):
    epoch_loss = 0
    
    for step in range(3):  # 3バッチ
        batch_input_ids = torch.randint(0, 1000, (4, 10))
        batch_labels = torch.randint(0, 2, (4,))
        
        loss = train_step(model, batch_input_ids, batch_labels, optimizer, criterion)
        epoch_loss += loss
    
    avg_loss = epoch_loss / 3
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/5 - Loss: {avg_loss:.4f}")

print("\n✅ 学習完了")

In [ ]:
# 損失の可視化
plt.figure(figsize=(10, 6))
plt.plot(losses, marker='o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"初期 Loss: {losses[0]:.4f}")
print(f"最終 Loss: {losses[-1]:.4f}")
print(f"改善率: {(1 - losses[-1] / losses[0]) * 100:.1f}%")

## セクション9: 演習問題

以下の演習に取り組んでみてください！

### 演習1: Causal Attention の実装

将来のトークンに注目できないようにマスクする Causal Attention を実装してみてください。

**ヒント**: softmax 前にマスクを適用します。

In [ ]:
def causal_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor
):
    """
    Causal Attention を計算します（将来のトークンが見えない）
    
    TODO: ここに実装を書いてください
    """
    d_k = Q.shape[-1]
    seq_len = Q.shape[-2]
    
    # スコア計算
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # TODO: Causal マスクを作成してスコアに適用
    # ヒント: torch.triu() を使用
    
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

print("TODO: Causal Attention を実装してください")

### 演習2: Attention ウェイトの分析

異なるシーケンスに対して Attention パターンがどう変わるか観察してください。

In [ ]:
# 異なる文章での Attention パターンを比較
sentences = [
    "I love cats",
    "The quick brown fox",
    "attention is all you need"
]

fig, axes = plt.subplots(1, len(sentences), figsize=(15, 4))

for idx, sentence in enumerate(sentences):
    tokens = sentence.split()
    
    # Attention 計算
    embeddings = torch.randn(len(tokens), 8)
    output, weights = scaled_dot_product_attention(
        embeddings.unsqueeze(0),
        embeddings.unsqueeze(0),
        embeddings.unsqueeze(0)
    )
    
    # 可視化
    weights_np = weights[0].detach().numpy()
    sns.heatmap(weights_np, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=tokens, yticklabels=tokens,
                cbar=False, ax=axes[idx])
    axes[idx].set_title(f"'{sentence}'")

plt.tight_layout()
plt.show()

## セクション10: 次のステップ

このノートブックで学んだことを、以下のタスクに応用してみてください:

1. **テキスト分類**: 異なるデータセットで SimpleTransformer をファインチューニング
2. **機械翻訳**: Encoder-Decoder アーキテクチャに拡張
3. **質問応答**: BERT のような Pre-training - Fine-tuning パイプライン
4. **効率最適化**: Linear Attention や Flash Attention の実装
5. **ビジョンタスク**: Vision Transformer (ViT) の実装

---

**参考資料**:
- 「Attention is All You Need」論文
- Hugging Face Transformers ドキュメント
- PyTorch 公式チュートリアル

Happy Learning! 🚀